In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/filtered_complaints.csv"
)

print(df.shape)
df.head()

(353769, 20)


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,word_count,cleaned_text
0,2025-06-13,Credit card,Store credit card,Getting a credit card,Card opened without my consent or knowledge,A XXXX XXXX card was opened under my name by a...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",TX,78230,Servicemember,Consent provided,Web,2025-06-13,Closed with non-monetary relief,Yes,NaN,14069121,91,a xxxx xxxx card was opened under my name by a...
1,2025-06-13,Checking or savings account,Checking account,Managing an account,Deposits and withdrawals,I made the mistake of using my wellsfargo debi...,Company has responded to the consumer and the ...,WELLS FARGO & COMPANY,ID,83815,NaN,Consent provided,Web,2025-06-13,Closed with explanation,Yes,NaN,14061897,109,i made the mistake of using my wellsfargo debi...
2,2025-06-12,Credit card,General-purpose credit card or charge card,"Other features, terms, or problems",Other problem,"Dear CFPB, I have a secured credit card with c...",Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,11220,NaN,Consent provided,Web,2025-06-13,Closed with monetary relief,Yes,NaN,14047085,156,dear cfpb i have a secured credit card with ci...
3,2025-06-12,Credit card,General-purpose credit card or charge card,Incorrect information on your report,Account information incorrect,I have a Citi rewards cards. The credit balanc...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",IL,60067,NaN,Consent provided,Web,2025-06-12,Closed with explanation,Yes,NaN,14040217,233,i have a citi rewards cards the credit balance...
4,2025-06-09,Credit card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,b'I am writing to dispute the following charge...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",TX,78413,Older American,Consent provided,Web,2025-06-09,Closed with monetary relief,Yes,NaN,13968411,454,b i am writing to dispute the following charge...


In [2]:
df.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID', 'word_count',
       'cleaned_text'],
      dtype='object')

In [3]:
df["Product"].value_counts()

Product
Checking or savings account                                140319
Money transfer, virtual currency, or money service          97188
Credit card                                                 80667
Payday loan, title loan, or personal loan                   17238
Consumer Loan                                                9461
Payday loan, title loan, personal loan, or advance loan      8896
Name: count, dtype: int64

In [4]:
df["Product"].value_counts(normalize=True)

Product
Checking or savings account                                0.396640
Money transfer, virtual currency, or money service         0.274722
Credit card                                                0.228022
Payday loan, title loan, or personal loan                  0.048727
Consumer Loan                                              0.026743
Payday loan, title loan, personal loan, or advance loan    0.025146
Name: proportion, dtype: float64

Creating a Stratified Sample

In [5]:
sample_size = 12000

sample_df = (
    df.groupby("Product", group_keys=False)
      .apply(
          lambda x: x.sample(
              frac=sample_size / len(df),
              random_state=42
          )
      )
)

print(sample_df.shape)

(12001, 20)


/var/folders/43/1hrrs9l51tg3wckl1gxm1mrh0000gn/T/ipykernel_23162/669828748.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("Product", group_keys=False)


In [6]:
sample_df["Product"].value_counts(normalize=True)

Product
Checking or savings account                                0.396634
Money transfer, virtual currency, or money service         0.274727
Credit card                                                0.227981
Payday loan, title loan, or personal loan                  0.048746
Consumer Loan                                              0.026748
Payday loan, title loan, personal loan, or advance loan    0.025165
Name: proportion, dtype: float64

Preparing the Text Column

In [7]:
sample_df["cleaned_text"] = (
    sample_df["Consumer complaint narrative"]
    .fillna("")
    .astype(str)
)

Chunking

In [8]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

In [9]:
sample_text = sample_df["cleaned_text"].iloc[0]

chunks = text_splitter.split_text(sample_text)

print(len(chunks))
print(chunks[0][:200])

1
Bank located in has received my check of {$1400.00} showed no proof of fraudulent volunteering closing out my accounts while in Her presenc. Walked out with {$200.00}, instead of {$1600.00}, closed ac


Creating Chunk Dataset

In [10]:
chunk_records = []

for _, row in sample_df.iterrows():

    chunks = text_splitter.split_text(
        row["cleaned_text"]
    )

    for i, chunk in enumerate(chunks):

        chunk_records.append({
            "complaint_id": row["Complaint ID"],
            "product": row["Product"],
            "chunk_index": i,
            "text": chunk
        })

In [11]:
chunk_df = pd.DataFrame(
    chunk_records
)

print(chunk_df.shape)

(38652, 4)


Generate Embeddings

In [12]:
import numpy as np
print(np.__version__)

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("OK")

1.26.4
OK


In [13]:
texts = chunk_df["text"].tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True
)

Batches:   0%|          | 0/1208 [00:00<?, ?it/s]

In [14]:
embeddings.shape

(38652, 384)

Creating ChromaDB Vector Store

In [15]:
import chromadb

In [16]:
client = chromadb.PersistentClient(
    path="../vector_store/chroma_db"
)

In [17]:
collection = client.get_or_create_collection(
    name="complaints"
)

Store Chunks and Metadata

In [18]:

import chromadb

client = chromadb.PersistentClient(path="./vector_store")

collection = client.get_or_create_collection("complaints")


In [19]:
collection.count()

3300

Test Retrieval

In [20]:
query = (
    "customers unhappy with credit cards"
)

In [21]:
query_embedding = model.encode(
    query
)

In [22]:
results = collection.query(
    query_embeddings=[
        query_embedding.tolist()
    ],

    n_results=5
)

In [23]:
results["documents"][0]

['for its consumers where nothing is done and loyal people like myself who have been with the company and also have relations with other banking institutions with no issues and dont see the same problem exist, go unheard or hear the excuses the customer service representatives have to apply as a cover where the underlying issues continue to get overlooked and continue to cause ongoing problems and stress for the consumers!!!',
 "I have visited the bank several times to speak with bank manager regarding this issue. I have not received a direct answer. The manager and representation at this branch are rude to me. The bank manager even told me not to apply for a credit card because she doesn't think my score would be high enough. I believe the bank manager wanted to discourage me from applying for a credit card due to XXXX discrimination.",
 'A total of 29 unauthorized charges, and another 34 amounting to {$1200.00}, and {$1600.00} were made on my cards starting XXXX XXXX XXXX Despite fil

In [24]:
len(embeddings)

38652

In [25]:
chunk_df.head()

,complaint_id,product,chunk_index,text
0,11750738,Checking or savings account,0,Bank located in has received my check of {$140...
1,7166095,Checking or savings account,0,Overdraft fees being charged and ATM fees bein...
2,8225373,Checking or savings account,0,I am writing to formally submit a complaint ag...
3,8225373,Checking or savings account,1,"In the initial dispute ( Claim # XXXX ), Capit..."
4,8225373,Checking or savings account,2,"Regrettably, the situation escalated when I di..."


In [26]:
client.list_collections()

[Collection(name=complaints)]

In [27]:
len(chunk_records)

38652

In [28]:
chunk_records[0]

{'complaint_id': 11750738,
 'product': 'Checking or savings account',
 'chunk_index': 0,
 'text': 'Bank located in has received my check of {$1400.00} showed no proof of fraudulent volunteering closing out my accounts while in Her presenc. Walked out with {$200.00}, instead of {$1600.00}, closed accounts,, showed no proof to me, that my check was fraudulent, stealing my money from work study, I received from a work study Job through XXXX XXXX XXXX XXXX XXXX. I was treated UNFAIRLY. They didnt give None of documents. Just word of mouth'}

In [29]:
'embeddings' in globals()

True

In [30]:
len(embeddings)

38652

In [31]:
'embeddings' in globals()

True

In [32]:
'collection' in globals()

True

In [33]:
collection.count()

3300

In [34]:
ids = [str(i) for i in range(len(chunk_records))]

In [36]:
batch_size = 500

for i in range(0, len(chunk_records), batch_size):

    batch_records = chunk_records[i:i+batch_size]
    batch_embeddings = embeddings[i:i+batch_size]

    collection.add(
        ids=[
            f"{x['complaint_id']}_{x['chunk_index']}"
            for x in batch_records
        ],

        embeddings=batch_embeddings,

        documents=[
            x["text"]
            for x in batch_records
        ],

        metadatas=[
            {
                "complaint_id": int(x["complaint_id"]),
                "product": str(x["product"]),
                "chunk_index": int(x["chunk_index"])
            }
            for x in batch_records
        ]
    )

    print(f"Inserted batch {i} → {i + len(batch_records)}")

Inserted batch 0 → 500
Inserted batch 500 → 1000
Inserted batch 1000 → 1500
Inserted batch 1500 → 2000
Inserted batch 2000 → 2500
Inserted batch 2500 → 3000
Inserted batch 3000 → 3500
Inserted batch 3500 → 4000
Inserted batch 4000 → 4500
Inserted batch 4500 → 5000
Inserted batch 5000 → 5500
Inserted batch 5500 → 6000
Inserted batch 6000 → 6500
Inserted batch 6500 → 7000
Inserted batch 7000 → 7500
Inserted batch 7500 → 8000
Inserted batch 8000 → 8500
Inserted batch 8500 → 9000
Inserted batch 9000 → 9500
Inserted batch 9500 → 10000
Inserted batch 10000 → 10500
Inserted batch 10500 → 11000
Inserted batch 11000 → 11500
Inserted batch 11500 → 12000
Inserted batch 12000 → 12500
Inserted batch 12500 → 13000
Inserted batch 13000 → 13500
Inserted batch 13500 → 14000
Inserted batch 14000 → 14500
Inserted batch 14500 → 15000
Inserted batch 15000 → 15500
Inserted batch 15500 → 16000
Inserted batch 16000 → 16500
Inserted batch 16500 → 17000
Inserted batch 17000 → 17500
Inserted batch 17500 → 18000
